# Generate Classification Charts

This notebook loads classified complaint data and generates visualization charts.

**Author**: Nikita Walvekar (walvekarn)

**Outputs**:
- `charts/complaint_categories.png`
- `charts/confidence_distribution.png`
- `charts/routing_breakdown.png`
- `charts/severity_distribution.png`


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path


In [ ]:
# Load the classified data
data_path = Path('../data/outputs/classified_sample_100.json')
with open(data_path, 'r') as f:
    data = json.load(f)

complaints = data['complaints']
print(f"Loaded {len(complaints)} complaints")


In [ ]:
# Extract classification data
products = [c['classification']['product'] for c in complaints]
confidences = [c['classification']['confidence'] for c in complaints]
routing = [c['classification']['recommended_routing'] for c in complaints]
severities = [c['classification']['severity_score'] for c in complaints]

print(f"Extracted {len(products)} product classifications")
print(f"Unique products: {len(set(products))}")


## 1. Complaint Categories (Product Distribution)


In [ ]:
# Count products
product_counts = Counter(products)
sorted_products = sorted(product_counts.items(), key=lambda x: x[1], reverse=True)

# Take top 10 or all if less
top_n = min(10, len(sorted_products))
labels = [p[0] for p in sorted_products[:top_n]]
values = [p[1] for p in sorted_products[:top_n]]

# Create bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(labels)), values, color='#2E86AB', alpha=0.8)
plt.xlabel('Product Category', fontsize=12, fontweight='bold')
plt.ylabel('Number of Complaints', fontsize=12, fontweight='bold')
plt.title('Complaint Distribution by Product Category', fontsize=14, fontweight='bold', pad=20)
plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, values)):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{value}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
output_path = Path('../charts/complaint_categories.png')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()


## 2. Confidence Distribution


In [ ]:
# Bin confidences
high_conf = sum(1 for c in confidences if c > 0.9)
medium_conf = sum(1 for c in confidences if 0.75 <= c <= 0.9)
low_conf = sum(1 for c in confidences if c < 0.75)

labels = ['High\n(>0.9)', 'Medium\n(0.75-0.9)', 'Low\n(<0.75)']
values = [high_conf, medium_conf, low_conf]
colors = ['#06A77D', '#FFB627', '#D62828']

# Create bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(labels, values, color=colors, alpha=0.8)
plt.xlabel('Confidence Level', fontsize=12, fontweight='bold')
plt.ylabel('Number of Complaints', fontsize=12, fontweight='bold')
plt.title('Classification Confidence Distribution', fontsize=14, fontweight='bold', pad=20)
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for bar, value in zip(bars, values):
    height = bar.get_height()
    percentage = (value / len(complaints)) * 100
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{value}\n({percentage:.0f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
output_path = Path('../charts/confidence_distribution.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()


## 3. Routing Breakdown


In [ ]:
# Count routing destinations
routing_counts = Counter(routing)
sorted_routing = sorted(routing_counts.items(), key=lambda x: x[1], reverse=True)

labels = [r[0] for r in sorted_routing]
values = [r[1] for r in sorted_routing]
percentages = [(v / len(complaints)) * 100 for v in values]

# Create pie chart
plt.figure(figsize=(10, 8))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#06A77D', '#7209B7', '#FFB627']
explode = [0.05 if i == 0 else 0 for i in range(len(labels))]

wedges, texts, autotexts = plt.pie(values, 
                                     labels=labels,
                                     autopct='%1.0f%%',
                                     startangle=90,
                                     colors=colors[:len(labels)],
                                     explode=explode,
                                     shadow=True,
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})

# Make percentage text more visible
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

plt.title('Complaint Routing Distribution by Department', fontsize=14, fontweight='bold', pad=20)
plt.axis('equal')
plt.tight_layout()

output_path = Path('../charts/routing_breakdown.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()


## 4. Severity Distribution


In [ ]:
# Bin severities
severity_bins = {
    'Low\n(1-3)': sum(1 for s in severities if 1 <= s <= 3),
    'Medium\n(4-5)': sum(1 for s in severities if 4 <= s <= 5),
    'High\n(6-7)': sum(1 for s in severities if 6 <= s <= 7),
    'Critical\n(8-10)': sum(1 for s in severities if 8 <= s <= 10)
}

labels = list(severity_bins.keys())
values = list(severity_bins.values())
colors = ['#06A77D', '#FFB627', '#F18F01', '#D62828']

# Create bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(labels, values, color=colors, alpha=0.8)
plt.xlabel('Severity Level', fontsize=12, fontweight='bold')
plt.ylabel('Number of Complaints', fontsize=12, fontweight='bold')
plt.title('Complaint Severity Distribution', fontsize=14, fontweight='bold', pad=20)
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for bar, value in zip(bars, values):
    height = bar.get_height()
    percentage = (value / len(complaints)) * 100
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{value}\n({percentage:.0f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
output_path = Path('../charts/severity_distribution.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved: {output_path}")
plt.show()


## Summary

All charts generated successfully!


In [ ]:
print("\n" + "="*60)
print("CHART GENERATION COMPLETE")
print("="*60)
print("\nGenerated charts:")
print("  ✓ charts/complaint_categories.png")
print("  ✓ charts/confidence_distribution.png")
print("  ✓ charts/routing_breakdown.png")
print("  ✓ charts/severity_distribution.png")
print("\nAll visualizations ready for README!")
